<a href="https://colab.research.google.com/github/ngynth/Spatial_Audio_Codec/blob/main/spatial_audio_codec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Update package lists and install ffmpeg
!apt-get update && apt-get install -y ffmpeg

# Install Python dependencies
!pip install numpy scipy matplotlib scikit-learn soundfile streamlit

In [ ]:
%%writefile compressor.py
import numpy as np
from sklearn.decomposition import PCA

class SpatialCodec:
    def __init__(self, n_components=2):
        self.n_components = n_components
        self.pca = PCA(n_components=n_components)

    def compress(self, audio_data):
        return self.pca.fit_transform(audio_data)

    def decompress(self, compressed_data):
        return self.pca.inverse_transform(compressed_data)

    def get_compression_ratio(self, original_shape):
        #Calculate number of samples in original and compressed sounds
        orig_size = original_shape[0] * original_shape[1]
        comp_size = original_shape[0] * self.n_components
        return orig_size / comp_size

In [ ]:
%%writefile spatial_utils.py
import numpy as np
from scipy import signal

def calculate_energy_vector(data):
    """Calculates 3D magnitude, azimuth, and elevation."""
    # Handle both 4-channel (Ambisonic) and 2-channel (PCA components)
    if data.shape[1] >= 4:
        w, x, y, z = data[:, 0], data[:, 1], data[:, 2], data[:, 3]
        energy_w = np.sum(w**2) + 1e-10
        # Coordinate of energy vector
        r_x, r_y, r_z = np.sum(x*w)/energy_w, np.sum(y*w)/energy_w, np.sum(z*w)/energy_w
    else:
        # Proxy for Compressed Azimuth using first two PCA components
        r_x, r_y, r_z = np.mean(data[:, 0]), np.mean(data[:, 1]), 0

    magnitude = np.sqrt(r_x**2 + r_y**2 + r_z**2)
    azimuth = np.arctan2(r_y, r_x)
    elevation = np.arcsin(np.clip(r_z / (magnitude + 1e-10), -1, 1))
    return magnitude, azimuth, elevation

def get_3d_perceptual_metrics(orig, comp, decomp, fs):
    # Calculate energy envelopes (original, compressed, decompressed)
    o_e = np.sqrt(np.sum(orig**2, axis=1))
    c_e = np.sqrt(np.sum(comp**2, axis=1)) * (np.max(o_e)/(np.max(comp)+1e-10))
    d_e = np.sqrt(np.sum(decomp**2, axis=1))

    # Calculates metrics across all three stages of the codec.
    freqs, psd = signal.welch(o_e, fs, nperseg=1024)
    mask_thresh = psd * 0.05 + 1e-12
    return o_e, c_e, d_e, freqs, psd, mask_thresh

def calculate_moving_cues(binaural_data, frame_size=1024):
    left, right = binaural_data[:, 0], binaural_data[:, 1]
    itds, ilds = [], []
    for i in range(0, len(left) - frame_size, frame_size):
        l_seg, r_seg = left[i:i+frame_size], right[i:i+frame_size]
        ild = 20 * np.log10((np.std(l_seg) + 1e-10) / (np.std(r_seg) + 1e-10))
        corr = signal.correlate(l_seg, r_seg, mode='same')
        itd = np.argmax(corr) - (frame_size // 2)
        itds.append(itd); ilds.append(ild)
    return np.array(itds), np.array(ilds)

def simple_binaural_render(data):
    left = (data[:, 0] + data[:, 2]) * 0.707
    right = (data[:, 0] - data[:, 2]) * 0.707
    return np.stack([left, right], axis=1)

def calculate_snr(orig, decomp):
    noise = orig - decomp
    return 10 * np.log10(np.sum(orig**2) / (np.sum(noise**2) + 1e-10))

In [ ]:
%%writefile visualization.py
import matplotlib.pyplot as plt
import numpy as np

def plot_waveforms_and_spectrograms(data, compressed, decompressed, fs, labels, save_path):
    """Generates the 4x2 waveform and spectrogram grid."""
    fig, axs = plt.subplots(4, 2, figsize=(15, 20))

    for i in range(4):
        # Column 1: Waveforms
        axs[i, 0].plot(data[:1000, i], label='Original', alpha=0.4, color='blue')
        comp_idx = 0 if i < 2 else 1
        axs[i, 0].plot(compressed[:1000, comp_idx], label=f'Compressed (PC{comp_idx+1})', color='green', alpha=0.6, linewidth=1)
        axs[i, 0].plot(decompressed[:1000, i], label='Decompressed', color='orange', alpha=0.8)

        axs[i, 0].set_title(f"Channel {labels[i]} Waveform Transformation")
        axs[i, 0].set_xlabel("Time (s)")
        axs[i, 0].set_ylabel("Amplitude")
        axs[i, 0].legend(loc='upper right')

        # Column 2: Difference Spectrograms
        diff = data[:, i] - decompressed[:, i]
        axs[i, 1].specgram(diff + 1e-10, Fs=fs, NFFT=1024, cmap='magma')
        axs[i, 1].set_title(f"Channel {labels[i]} Difference Spectrogram")
        axs[i, 1].set_xlabel("Time (s)")
        axs[i, 1].set_ylabel("Frequency (Hz)")

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close(fig) # Close to free up memory

def plot_perceptual_analysis(metrics, binaural_cues, spatial_data, save_path):
    """Generates the 2x2 perceptual and localization analysis grid."""
    # Unpack metrics
    o_e, c_e, d_e, freqs, psd, thresh = metrics
    itds, ilds = binaural_cues
    mag_o, az_o, mag_d, az_d, el_d = spatial_data

    fig = plt.figure(figsize=(16, 14))

    # 1. Masking Threshold
    ax_m = fig.add_subplot(2, 2, 1)
    ax_m.semilogy(freqs, psd + 1e-10, label='Signal PSD')
    ax_m.semilogy(freqs, thresh + 1e-10, 'r:', label='Masking Threshold')
    ax_m.set_title("Directional Masking Analysis")
    ax_m.set_xlabel("Frequency (Hz)"); ax_m.set_ylabel("Power/Freq (dB/Hz)"); ax_m.legend()

    # 2. Total Acoustic Energy
    ax_e = fig.add_subplot(2, 2, 2)
    ax_e.plot(o_e[:1500], label='Original', alpha=0.6)
    ax_e.plot(c_e[:1500], label='Compressed (Scaled)', alpha=0.8)
    ax_e.plot(d_e[:1500], label='Decompressed', alpha=0.6)
    ax_e.set_title("Total Acoustic Energy Path (L2 Norm)")
    ax_e.set_xlabel("Time (Samples)"); ax_e.set_ylabel("Energy Magnitude"); ax_e.legend()

    # 3. ITD & ILD
    ax_c = fig.add_subplot(2, 2, 3)
    jitter = np.random.normal(0, 1, size=len(itds))
    angles = np.full_like(itds, np.degrees(az_d)) + jitter
    ax_c.scatter(angles, itds, color='blue', alpha=0.4, label='ITD (s)')
    ax_c.scatter(angles, ilds, color='green', alpha=0.4, label='ILD (dB)')
    ax_c.set_xlim([-180, 180])
    ax_c.set_xlabel("Decompressed Azimuth (Degrees)"); ax_c.set_ylabel("Cue Value")
    ax_c.set_title("Binaural Cue Distribution (HRTF-Style)"); ax_c.legend()

    # 4. 3D Polar Vector
    ax_p = fig.add_subplot(2, 2, 4, projection='polar')
    ax_p.annotate('', xy=(az_o, mag_o), xytext=(0,0), arrowprops=dict(facecolor='blue', width=2))
    ax_p.annotate('', xy=(az_d, mag_d), xytext=(0,0), arrowprops=dict(facecolor='red', alpha=0.5, width=2))
    ax_p.text(az_o, mag_o + 0.15, 'Original', color='blue', ha='center', va='center', weight='bold')
    ax_p.text(az_d + 0.2, mag_d + 0.2, 'Decompressed', color='red', ha='right', va='center', weight='bold')
    ax_p.set_title(f"Spatial Localization (Elevation: {np.degrees(el_d):.1f}°)")

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close(fig)

In [ ]:
%%writefile main.py
import os, time
import numpy as np
import soundfile as sf
import matplotlib
matplotlib.use('Agg')
from compressor import SpatialCodec
import spatial_utils as utils
import visualization as viz

def run_project(input_file):
    print(f"--- Processing: {input_file} ---")
    data, fs = sf.read(input_file)
    if data.shape[1] < 4: raise ValueError("Input must be 4-channel Ambisonic")

    # 1. Processing
    start_time = time.time()
    codec = SpatialCodec(n_components=2)
    compressed = codec.compress(data)
    decompressed = codec.decompress(compressed)
    latency = (time.time() - start_time) * 1000
    binaural = utils.simple_binaural_render(decompressed)

    # 2. Rendering for Demo
    binaural_out = utils.simple_binaural_render(decompressed)
    # Save the binaural demo to the processed folder
    output_path = 'Output/speech_wav_output.wav'
    sf.write(output_path, binaural_out, fs)

    # 3. Metrics & Terminal Output
    mag_o, az_o, el_o = utils.calculate_energy_vector(data)
    mag_c, az_c, el_c = utils.calculate_energy_vector(compressed)
    mag_d, az_d, el_d = utils.calculate_energy_vector(decompressed)

    orig_br = (fs * 16 * 4) / 1000
    comp_br = (fs * 16 * 2) / 1000

    # Keeping terminal output identical
    print(f"\n--- Codec Metrics ---")
    print(f"Compression Ratio:    {codec.get_compression_ratio(data.shape):.1f}:1")
    print(f"Original Bitrate:   {orig_br:.1f} kbps")
    print(f"Compressed Bitrate: {comp_br:.1f} kbps")
    print(f"Decompressed Bitrate:      {comp_br:.1f} kbps")
    print(f"SNR:                  {utils.calculate_snr(data, decompressed):.2f} dB")
    print(f"Latency:              {latency:.2f} ms")
    print(f"Original Azimuth:     {np.degrees(az_o):.1f}°")
    print(f"Compressed Azimuth:   {np.degrees(az_c):.1f}°")
    print(f"Decompressed Azimuth: {np.degrees(az_d):.1f}°")

    # --- IMAGE 1: Waveforms & Difference Spectrograms ---
    labels = ['W (Omni)', 'X (Front)', 'Y (Left)', 'Z (Up)']
    path1 = 'Output/output_image_1.png'
    viz.plot_waveforms_and_spectrograms(data, compressed, decompressed, fs, labels, path1)

    # --- IMAGE 2: Perceptual & Localization Analysis ---
    metrics = utils.get_3d_perceptual_metrics(data, compressed, decompressed, fs)
    binaural_cues = utils.calculate_moving_cues(binaural)
    spatial_data = (mag_o, az_o, mag_d, az_d, el_d)
    path2 = 'Output/output_image_2.png'
    viz.plot_perceptual_analysis(metrics, binaural_cues, spatial_data, path2)

    print(f"--- Reports saved to Output/ ---")

if __name__ == "__main__":
    run_project('Datasets/speech.wav')

In [ ]:
# After wav file analysis, convert the file to aac then converted back to wav
!ffmpeg -i Datasets/speech.wav -c:a aac -b:a 384k Datasets/speech_aac.aac
!ffmpeg -i Datasets/speech_aac.aac Datasets/speech_aac_decoded.wav

In [ ]:
# First run: For original wav input (skip the above terminal command)
# Second run: Run the above terminal command. For acc-to-wav input (e.g speech_aac_decoded.wav)
# CHANGE the name of input/output in second run. Overwrite main.py file
"""
e.g
    output_path = 'Output/speech_aac_output.wav'
    run_project('Datasets/speech_aac_decoded.wav')
    path1 = 'Output/output_image_3.png'
    path2 = 'Output/output_image_4.png'
"""
!python main.py